# Wheeled Mobile Robots

Source orientation: printed pages 513-564, PDF pages 531-582. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

How do wheel designs decide which chassis velocities are directly available?

This question is the thread for the chapter. The goal is not to memorize a list of formulas; it is to make the geometry inspectable. A robot mechanism is a physical machine, but the way we reason about it is through spaces, maps, constraints, tangents, and dual forces. The notebook keeps those objects visible. Every diagram and computation below is designed to answer a local question that a reader can check: what is moving, what is fixed, what map is being applied, what invariant should survive, and where can the model fail?

## Translation Guide

The source chapter or appendix is translated into computational language using these terms: SE(2), wheel constraint, omniwheel, mecanum wheel, nonholonomic, odometry, mobile manipulation. The important conversions for this notebook are:

- Wheel rolling constraints map chassis twists to wheel speeds.
- Omnidirectional bases span planar velocity directly.
- Nonholonomic bases can reach poses by paths, not arbitrary instantaneous sideways motion.

The central habit is to name the representation and the invariant at the same time. Coordinates are useful only when we know what geometric object they represent. A column of joint angles may describe a point on a torus, a homogeneous transform may describe a rigid frame, and a matrix rank may reveal an instantaneous loss of motion. The notebook therefore pairs each formula with a small experiment: a plot, a residual, an ellipsoid, a path, a graph, or a constraint surface.

## Route Through the Notebook

1. Establish the objects and conventions needed for wheeled mobile robots.
2. Build a concept dependency map so definitions have visible structure.
3. Generate the main visual lab: omnidirectional and nonholonomic mobile bases.
4. Run a compact worked example that exposes an invariant.
5. Try an applied lab that changes a parameter and asks what should remain true.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/chapter-13-wheeled-mobile-robots/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| omnidirectional and nonholonomic mobile bases | unicycle rollout and wheel velocity map | `artifacts/chapter-13-wheeled-mobile-robots/figures/wheeled-mobile-robots-lab.png` | how wheel constraints shape reachable velocity and path geometry |
| numeric invariant check | residual bar chart and JSON summary | `artifacts/chapter-13-wheeled-mobile-robots/figures/wheeled-mobile-robots-checks.png` | small residuals, positive margins, or rank changes |

The first visual is a dependency map. It is intentionally simple: before computing anything, the reader should see which concepts support which later moves. The second visual is the main lab for this chapter. It turns the chapter's core geometry into something that can be inspected rather than imagined. The third visual is a numeric check: a residual, rank, eigenvalue, path length, or comparable margin that can be tested after the figure is built.

## Working Principles

The most reliable way to learn robotics geometry is to move between three views. The first view is symbolic: equations name maps and constraints. The second view is numerical: a small instance exposes scale, rank, conditioning, and residuals. The third view is visual: geometry becomes a shape, path, ellipsoid, cone, graph, or surface. This notebook keeps all three views close together. If a symbolic statement is correct, the numerical experiment should produce the expected small residual or positive margin. If a visual is meaningful, it should make a specific invariant or failure mode easier to see.

For wheeled mobile robots, the relevant failure modes are not side details; they are part of the concept. Singularities, chart boundaries, rank loss, time-scaling limits, contact mode changes, and wheel constraints are all examples of geometry asserting itself. A robust robotics model does not pretend those edges are absent. It names them, draws them, and then tests a small case so the reader can recognize the issue in later code.

## Reading Wheel Constraints

A wheeled mobile robot is easiest to read as a set of velocity constraints on a planar rigid body. The chassis pose lives in SE(2), and a body-frame chassis twist can be written as `(omega, v_x, v_y)`: yaw rate, forward velocity, and lateral velocity. A wheel does not merely add an actuator. It also declares which component of ground contact is allowed to roll, which component should not slip, and which component is converted into wheel angular speed. That is why the same rectangular chassis can behave like several different robots after only the wheel modules change.

For a mecanum or omni base, the roller geometry lets four wheel speeds span all three planar twist components. In matrix language the wheel map `u = H V_b` has rank three, so a least-squares inverse can recover the commanded chassis twist from compatible wheel speeds. This does not mean the base is immune to saturation, compliance, or floor friction; it means the ideal instantaneous kinematic model has no missing planar direction. A differential-drive or unicycle model is different. It can command forward speed and yaw rate, but its body-sideways velocity is constrained to zero at each instant. The robot may still reach poses with sideways displacement by following curved paths, so global reachability and instantaneous mobility must be kept separate.

Odometry is the running account of those small twist decisions. The integration step rotates body-frame motion into the world frame, accumulates heading, and quietly amplifies bias if wheel radii, roller angles, or encoder signs are wrong. The checks below therefore look at both algebra and motion: the wheel matrix should have the expected rank, pseudo-inversion should reconstruct compatible twists, and a unicycle rollout should produce a nonzero path with a heading change that matches the chosen controls.

## Applied Lab

Roll out a unicycle trajectory and compare it with a mecanum wheel velocity map.

In the lab, vary one parameter at a time and predict the direction of change before running the code. For example, ask whether a rank should change, whether a path should lengthen, whether an eigenvalue should stay positive, whether a cone should widen, or whether a coordinate chart should approach a singular value. This prediction step is small, but it is what turns a figure into a mathematical instrument.

## Pitfalls To Watch

- A car-like robot can be controllable while still unable to move sideways instantly.
- Odometry integrates small errors into pose drift.

These pitfalls are deliberately operational. If a computation becomes confusing, check frame labels, units, rank, and the distinction between a physical object and its coordinates. Many robotics errors are not arithmetic mistakes; they are mismatches between a representation and the geometry it claims to encode.

## Takeaways

- Mobile robot kinematics is constraint geometry on SE(2).
- Instantaneous mobility and global reachability are different questions.

By the end of this notebook, the reader should be able to explain the chapter's main object, build a small computed example, interpret the generated visual artifact, and state at least one numerical sanity check that would catch an implementation mistake.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 13,
  "slug": "chapter-13-wheeled-mobile-robots",
  "title": "Wheeled Mobile Robots",
  "notebook": "13-wheeled-mobile-robots.ipynb",
  "printed_start": 513,
  "printed_end": 564,
  "pdf_start": 531,
  "pdf_end": 582,
  "part_slug": "part-04-control-contact-and-mobile-robots",
  "part_title": "Control, Contact, and Mobile Robots",
  "theme": "mobile",
  "visual_focus": "omnidirectional and nonholonomic mobile bases",
  "visual_kind": "unicycle rollout and wheel velocity map",
  "artifact_stem": "wheeled-mobile-robots",
  "inspection_target": "how wheel constraints shape reachable velocity and path geometry",
  "question": "How do wheel designs decide which chassis velocities are directly available?",
  "terms": [
    "SE(2)",
    "wheel constraint",
    "omniwheel",
    "mecanum wheel",
    "nonholonomic",
    "odometry",
    "mobile manipulation"
  ],
  "translation": [
    "Wheel rolling constraints map chassis twists to wheel speeds.",
    "Omnidirectional bases span planar velocity directly.",
    "Nonholonomic bases can reach poses by paths, not arbitrary instantaneous sideways motion."
  ],
  "lab": "Roll out a unicycle trajectory and compare it with a mecanum wheel velocity map.",
  "pitfalls": [
    "A car-like robot can be controllable while still unable to move sideways instantly.",
    "Odometry integrates small errors into pose drift."
  ],
  "takeaways": [
    "Mobile robot kinematics is constraint geometry on SE(2).",
    "Instantaneous mobility and global reachability are different questions."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard

In [ ]:
from utils.visuals import build_chapter_visuals

outputs = build_chapter_visuals(CHAPTER)
for artifact in outputs['figures']:
    display_artifact(artifact, width=760)
outputs['metrics']

## Worked Example

The worked example below asks a question that belongs to this chapter: if an ideal mecanum base is given a compatible set of wheel speeds, can the chassis twist be recovered? The matrix `H` maps the body-frame chassis twist `(omega, v_x, v_y)` to four wheel angular rates. Rank three means the four wheels overconstrain, rather than underconstrain, planar motion; the pseudo-inverse then gives a consistent least-squares reconstruction when the wheel speeds came from the same model. The example checks forward motion, sideways motion, pure spin, and a mixed twist so the sign pattern is visible instead of hidden in a single number.

The same cell also keeps a small unicycle comparison nearby. A unicycle rollout is not allowed to command body-frame sideways velocity directly, yet its world-frame path can still develop lateral displacement after the heading changes. This contrast is the chapter's recurring distinction: the instantaneous velocity space is a local object, while the set of reachable poses is built over time.


In [ ]:
from utils.mobile import mecanum_wheel_matrix, unicycle_rollout

wheel_map = mecanum_wheel_matrix(length=0.35, width=0.25, radius=0.05)
commanded_twists = np.array([
    [0.0, 0.40, 0.00],   # straight forward
    [0.0, 0.00, 0.35],   # body-left strafe
    [0.55, 0.00, 0.00],  # in-place yaw
    [0.25, 0.30, -0.20], # combined motion
])
wheel_speeds = commanded_twists @ wheel_map.T
recovered_twists = wheel_speeds @ np.linalg.pinv(wheel_map).T
reconstruction_error = np.linalg.norm(recovered_twists - commanded_twists)

curved_controls = np.c_[np.ones(120) * 0.42, np.ones(120) * 0.32]
curved_path = unicycle_rollout(curved_controls, dt=0.05)
worked_example = {
    "wheel_map_rank": int(np.linalg.matrix_rank(wheel_map)),
    "wheel_map_condition": float(np.linalg.cond(wheel_map)),
    "max_abs_wheel_speed": float(np.max(np.abs(wheel_speeds))),
    "reconstruction_error": float(reconstruction_error),
    "unicycle_final_pose": curved_path[-1].round(4).tolist(),
    "unicycle_lateral_world_displacement": float(curved_path[-1, 2] - curved_path[0, 2]),
}
worked_example


## Applied Lab

The applied lab treats wheel geometry as an experiment rather than a table of cases. First, it samples a unicycle trajectory whose forward speed is nearly constant while yaw rate oscillates; the resulting path length and final heading are quantities the reader can predict from the controls. Second, it inspects the same mecanum wheel matrix used in the worked example and asks what happens to the commanded wheel rates for a pure sideways chassis twist. Large or small wheel speeds are not good or bad by themselves; they are a scale check that must be interpreted with wheel radius, body dimensions, and actuator limits.

Before running the cell, make two predictions. The mecanum rank should remain three because the roller geometry spans planar twists. The unicycle should accumulate world-frame sideways displacement even though its body-frame lateral velocity is zero, because heading changes rotate later forward motion into a different world direction. If either prediction fails, the likely bug is a swapped coordinate convention, an incorrect sign in the wheel map, or an odometry update that forgot which frame a velocity lives in.


In [ ]:
from utils.mobile import mecanum_wheel_matrix, unicycle_rollout

controls = np.c_[
    np.ones(100) * 0.50,
    0.35 * np.sin(np.linspace(0.0, 2.0 * np.pi, 100)),
]
path = unicycle_rollout(controls, dt=0.05)
xy_steps = np.diff(path[:, 1:3], axis=0)
path_distance = float(np.sum(np.linalg.norm(xy_steps, axis=1)))
heading_span = float(path[:, 0].max() - path[:, 0].min())

H = mecanum_wheel_matrix()
sideways_twist = np.array([0.0, 0.0, 0.25])
sideways_wheel_speeds = H @ sideways_twist
lab_summary = {
    "theme": CHAPTER["theme"],
    "wheel_map_rank": int(np.linalg.matrix_rank(H)),
    "sideways_wheel_speeds": sideways_wheel_speeds.round(4).tolist(),
    "path_distance": path_distance,
    "heading_span": heading_span,
    "final_pose": path[-1].round(4).tolist(),
}
lab_summary


## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also checks the chapter-specific numerical claims made above: the mecanum wheel map spans three planar twist components, compatible wheel speeds reconstruct the original body twist to floating-point accuracy, and the unicycle rollout has a nonzero path length. The JSON summary under the chapter's artifact subtree is a machine-checkable trace of those claims.


In [ ]:
artifact_stats = {}
for artifact in outputs['figures']:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats['pixel_std'] > 2.0, f'low variation image: {resolved}'
    artifact_stats[resolved.name] = stats
assert_artifact(outputs['checks'], min_size=100)
assert worked_example['wheel_map_rank'] == 3
assert worked_example['reconstruction_error'] < 1e-10
assert lab_summary['wheel_map_rank'] == 3
assert lab_summary['path_distance'] > 0.1
sanity = {'chapter': CHAPTER['slug'], 'metrics': outputs['metrics'], 'worked_example': worked_example, 'lab_summary': lab_summary, 'artifact_stats': artifact_stats}
sanity_path = save_json(sanity, CHAPTER['slug'], 'checks', 'notebook-sanity.json')
display_artifact(sanity_path)
sanity
